In [27]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib as plt

print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"matplotlib version: {plt.__version__}")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
NumPy version: 2.0.2
pandas version: 2.3.3
scikit-learn version: 1.6.1
matplotlib version: 3.9.4


## Step2 Load and inspect

In [28]:
import pandas as pd

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=3132)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1158, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           60
building_type         0
ceiling_height_m    136
price_kzt             0
dtype: int64
                   count unique         top freq         mean         std  \
listing_id          1158   1080   ALM-10932    3          NaN         NaN   
listed_date         1158    246  2026-04-18   12          NaN         NaN   
district            1158     47       Medeu  147          NaN         NaN   
rooms               1158      6           2  440          NaN         NaN   
area_

In [29]:
print("Sample values of potential numeric columns")

for col in ["listed_date", "area_m2", "rooms", "floor", "total_floors","year_built", "ceiling_height_m", "price_kzt"]:
    if col in df.columns and df[col].dtype == "object":
        print(f"\n{col} samples:")
        print(df[col].sample(10, random_state=3132).tolist())
    

Sample values of potential numeric columns

listed_date samples:
['2026-02-23', '2026-01-30', '2026-01-31', '2026-05-18', '2026-01-27', '2026-02-10', '2026-04-25', '2026-03-11', '2026-07-11', '2026-05-29']

area_m2 samples:
['53.0', '105.9', '112.0 m²', '60.9', '58.8', '77.4', '57.8', '81.0', '77.5', '71.2']

rooms samples:
['2', '4', '4', '2', '2', '3', '2', '3', '3', '3']

price_kzt samples:
['35120000', '47130000', '125100000', '39190000', '59850000', '37960000', '47940000', '89 740 000 ₸', '48,480,000', '71100000']


In [30]:
print("Count duplicates")

exact_dups = df.duplicated().sum()
listing_dups = df.drop_duplicates()["listing_id"].duplicated().sum()

print(f"\n(a) Exact duplicate rows: {exact_dups}")
print(f"(b) Non-exact duplicates with same listing_id: {listing_dups}")

dup_listing_ids = df[df.duplicated(subset=["listing_id"], keep=False)]["listing_id"].drop_duplicates().iloc[:2]
print("\n--- Duplicate listing examples ---")
print(df[df["listing_id"].isin(dup_listing_ids)].sort_values("listing_id"))

Count duplicates

(a) Exact duplicate rows: 35
(b) Non-exact duplicates with same listing_id: 43

--- Duplicate listing examples ---
    listing_id listed_date district rooms area_m2  floor  total_floors  \
8    ALM-10003  2026-05-29   Auezov     2    48.6      2             5   
823  ALM-10003  2026-05-29   Auezov     2    48.6      2             5   
967  ALM-10003  2026-05-04   Auezov     2    48.6      2             5   
12   ALM-10938  2026-08-10    Medeu     5   133.0      1             9   
509  ALM-10938  2026-08-21    Medeu     5   133.0      1             9   

     year_built building_type  ceiling_height_m  price_kzt  
8        1978.0         panel              2.55   26120000  
823      1978.0         panel              2.55   26120000  
967      1978.0         panel              2.55   28320000  
12       2013.0      monolith              2.93  177470000  
509      2013.0      monolith              2.93  167850000  


### Step 2: Data Inspection Answers

#### 1.Types
pandas don't choose because area_m2, price_kzt should can be numbers but they have another symbols like kzt m2. for that reason pandas says they object

#### 2.Grain
duplicate listing_id but other attributes like price_kzt or listed_data different ,so it means same listing was reposted or updated with a new price or time.

#### 3.Validation Spec
| Column | Expected Type | Valid Range / Allowed Values |
| :--- | :--- | :--- |
| listing_id | Integer | Positive integer |
| listed_date | Date (datetime64) | 2026-05-01 to 2026-09-28 |
| district | Categorical / String |  Almaty districts: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu   |
| rooms | Integer | 1 to 5 |
| area_m2 | Float | 20.0 to 250.0 |
| floor | Integer | 1 to total_floors  |
| total_floors | Integer | 1 to 25 |
| year_built | Integer | 1950 to 2026 (0 is a sentinel, not a valid value) |
| building_type | Categorical / String |panel, brick, monolith |
| ceiling_height_m | Float | 2.2 to 4.0 m  |
| price_kzt | Float | 10,000,000 to 200,000,000 ₸ |

## Step 3: Clean and transform

In [31]:
cleaning_log = []

def log_step(operation_name, df_current):
    rows = len(df_current)
    cleaning_log.append({"Operation": operation_name, "Rows": rows})
    print(f"[{operation_name}] Rows count: {rows}")

log_step("Initial state (Step 2 output)", df)

[Initial state (Step 2 output)] Rows count: 1158


In [32]:
# 3a. Duplicates and Reposts
df = df.drop_duplicates().reset_index(drop=True)
log_step("3a. After dropping exact duplicates", df)

df["listed_date"] = pd.to_datetime(df["listed_date"], errors="coerce")

reposts_before = df.drop_duplicates()["listing_id"].duplicated().sum()
df = df.sort_values("listed_date").drop_duplicates(subset=["listing_id"], keep="last").reset_index(drop=True)
log_step("3a. After resolving reposts (kept latest date)", df)

print(f"\nReposts resolved: {reposts_before}")

[3a. After dropping exact duplicates] Rows count: 1123
[3a. After resolving reposts (kept latest date)] Rows count: 1080

Reposts resolved: 43


In [33]:
# 3b. District Categories Standardisation

print("\n--- Original District Values ---")
print(df["district"].value_counts(dropna=False))

def clean_district(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()
    
    if "алатау" in s or "alatau" in s: return "Alatau"
    if "алмал" in s or "almal" in s: return "Almaly"
    if "ауэз" in s or "auez" in s: return "Auezov"
    if "бостан" in s or "bostan" in s: return "Bostandyk"
    if "медеу" in s or "medeu" in s: return "Medeu"
    if "наурыз" in s or "nauryz" in s: return "Nauryzbay"
    if "турк" in s or "turk" in s: return "Turksib"
    if "жет" in s or "zhet" in s: return "Zhetysu"
    return val

df["district"] = df["district"].apply(clean_district)
log_step("3b. After district standardization", df)

print(f"\nUnique districts count: {df['district'].nunique()}")
print(f"Missing districts count: {df['district'].isna().sum()}")
print("Districts values:")
print(df["district"].value_counts())


--- Original District Values ---
district
Medeu            136
Bostandyk        124
Zhetysu          121
Auezov           119
Turksib          113
Almaly           108
Alatau           106
Nauryzbay         93
turksib           10
medeu             10
Наурызбайский      7
auezov             7
Nauryzbayskiy      6
Alatauskiy         6
Auezovskiy         6
 Zhetysu           6
AUEZOV             6
nauryzbay          5
 Turksib           5
Bostandykskiy      5
 Auezov            5
BOSTANDYK          5
ALMALY             5
bostandyk          4
 Medeu             4
Медеуский          4
NAURYZBAY          4
 Alatau            4
TURKSIB            4
almaly             4
Zhetysuskiy        3
MEDEU              3
Турксибский        3
 Almaly            3
Бостандыкский      3
 Bostandyk         3
ZHETYSU            3
Almalinskiy        3
Medeuskiy          2
Ауэзовский         2
zhetysu            2
ALATAU             2
Turksibskiy        2
alatau             1
Алатауский         1
Алмалинский 

In [34]:
# --- Step 3c: Text to Numbers Transformation  ---

# 1. rooms:
if df["rooms"].dtype == "object":
    df["rooms"] = df["rooms"].astype(str).str.strip().str.lower()
    df["rooms"] = df["rooms"].replace({"studio": "1", "студия": "1"})
    df["rooms"] = pd.to_numeric(df["rooms"].str.replace(r"\D", "", regex=True), errors="coerce")

# 2. area_m2:
if df["area_m2"].dtype == "object":
    df["area_m2"] = df["area_m2"].astype(str).str.replace("m²", "", regex=False)
    df["area_m2"] = df["area_m2"].str.replace("m2", "", regex=False)
    df["area_m2"] = df["area_m2"].str.replace(",", ".", regex=False).str.strip()
    df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")

# 3. price_kzt:
if df["price_kzt"].dtype == "object":
    df["price_kzt"] = df["price_kzt"].astype(str).str.replace("₸", "", regex=False)
    df["price_kzt"] = df["price_kzt"].str.replace("kzt", "", regex=False, case=False)
    df["price_kzt"] = df["price_kzt"].str.replace(r"\s+", "", regex=True)
    df["price_kzt"] = df["price_kzt"].str.replace(",", "", regex=False)
    df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")

# 4. ceiling_height_m:
if "ceiling_height_m" in df.columns and df["ceiling_height_m"].dtype == "object":
    df["ceiling_height_m"] = df["ceiling_height_m"].astype(str).str.replace("m", "", regex=False)
    df["ceiling_height_m"] = df["ceiling_height_m"].str.replace(",", ".", regex=False).str.strip()
    df["ceiling_height_m"] = pd.to_numeric(df["ceiling_height_m"], errors="coerce")

log_step("3c. After converting text columns to numeric", df)

print("\n--- Final Data Types (df.dtypes) ---")
print(df.dtypes)

print("--- Missing Values Check After Fix ---")
print(df[["rooms", "area_m2", "price_kzt"]].isna().sum())

[3c. After converting text columns to numeric] Rows count: 1080

--- Final Data Types (df.dtypes) ---
listing_id                  object
listed_date         datetime64[ns]
district                    object
rooms                        int64
area_m2                    float64
floor                        int64
total_floors                 int64
year_built                 float64
building_type               object
ceiling_height_m           float64
price_kzt                    int64
dtype: object
--- Missing Values Check After Fix ---
rooms        0
area_m2      0
price_kzt    0
dtype: int64
